# Notebook 2: Window Memory — `trim_messages()`
### (the modern version of `ConversationBufferWindowMemory`)

### The problem first
Notebook 1 keeps **everything**. Two problems as the chat grows:
1. **Money** — every turn resends the whole transcript to the API.
2. **Context limit** — eventually the transcript simply doesn't fit.

### What it does (simplest version)
Before calling the LLM, **keep only the most recent ~N tokens** and drop the rest.
The bot remembers the *last few minutes* of the conversation — and forgets the beginning.

### Human analogy
Talking to someone who only remembers the **last 5 minutes** of what you both said.

### Where to use it
- High-volume, casual bots (FAQ, chit-chat) where old turns don't matter.
- When you want a **hard, predictable token budget**.

### The trade-off (say it out loud in class)
Cheap and fast — **but the bot forgets old details**. That's exactly what the old
`ConversationBufferWindowMemory` did; we just do it with a function now.

### What it connects with later
If old details *do* matter → don't drop them, **compress** them. That's Notebook 3 (summary memory).

## Step 1 — Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup done.")

## Step 2 — See `trim_messages` with your own eyes (no graph yet)

`trim_messages` takes a list of messages and returns only the tail that fits the budget.
Run this and compare **before** vs **after**.

In [ ]:
fake_history = [
    HumanMessage("My name is Rahul."),
    AIMessage("Nice to meet you, Rahul!"),
    HumanMessage("Tell me fact number 1."),
    AIMessage("Honey never spoils. Archaeologists found edible 3000-year-old honey."),
    HumanMessage("Tell me fact number 2."),
    AIMessage("Octopuses have three hearts and blue blood."),
    HumanMessage("Tell me fact number 3."),
    AIMessage("Bananas are berries, but strawberries are not."),
]

trimmed = trim_messages(
    fake_history,
    strategy="last",                            # keep messages from the END
    token_counter=count_tokens_approximately,
    max_tokens=50,                              # tiny budget on purpose
    start_on="human",                           # never cut mid-turn
)

print(f"Before trim: {len(fake_history)} messages")
print(f"After trim:  {len(trimmed)} messages
")
for m in trimmed:
    print(f"  {m.type}: {m.content[:60]}")
# -> The early messages (including the name!) are gone from what the LLM would see.

## Step 3 — Put the window inside the agent

One line changes from Notebook 1: the node trims the state **before** calling the LLM.

**Important subtlety (students always miss this):**
the **checkpointer still stores the FULL history**. Trimming only controls *what the LLM sees this turn*.
Memory for storage ≠ memory for the prompt.

In [ ]:
def chatbot(state: MessagesState):
    recent = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=200,          # the "window size" — old WindowMemory's k, but in tokens
        start_on="human",
    )
    return {"messages": [llm.invoke(recent)]}   # LLM only sees the window

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile(checkpointer=InMemorySaver())
print("Agent ready with a 200-token window.")

## Step 4 — Watch it forget

We tell the bot our name, then push many long turns so the name **falls out of the window**.

In [ ]:
config = {"configurable": {"thread_id": "window-demo"}}

graph.invoke({"messages": [HumanMessage("My name is Rahul.")]}, config)
print("Turn 1 done — name stored in state AND inside the window (for now).")

In [ ]:
# Push the name out of the window with long turns
for i in range(8):
    graph.invoke(
        {"messages": [HumanMessage(
            f"Tell me an interesting science fact number {i}, with a full explanation."
        )]},
        config,
    )
    print(f"...turn {i + 2} done")

In [ ]:
out = graph.invoke({"messages": [HumanMessage("What is my name?")]}, config)
print("AI:", out["messages"][-1].content)
# -> Expected: it does NOT know. "My name is Rahul" fell out of the 200-token window.

## Try it yourself

1. Increase `max_tokens` from 200 to 2000 — does the name survive now? Why?
2. Print `len(out["messages"])` — the state still has **every** message! The checkpointer
   never forgot; we just hid old messages from the LLM.
3. Discussion: for a **banking** chatbot, is a window a good idea? What could go wrong?

**Next notebook:** what if we want old details to survive *without* paying for full history?
Answer: compress them into a summary → Notebook 3.